In [2]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import sequence
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input,Embedding,Bidirectional,LSTM,Dense,Dropout,Layer)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [3]:
df = pd.read_csv("C:\Projects\Ticketing system\customer_support_tickets_processed.csv")
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,business_category,lables,translated
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN,IT / Technical Support,0,"Dear Support Team,\n\nich would like to report..."
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,IT / Technical Support,0,"Dear Customer Support Team,\n\nI am writing to..."
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,Customer Service,1,"Dear Customer Support Team,\n\nI hope this mes..."
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,Billing / Finance,2,"Dear Customer Support Team,\n\nI hope this mes..."
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,Customer Service,1,"Dear Support Team,\n\nI hope this message reac..."


In [4]:

df["translated"].isna().value_counts()

translated
False    49778
True         2
Name: count, dtype: int64

In [5]:
df["translated"] = (df["translated"].fillna("").astype(str).str.strip())
df = df[df["translated"] != ""]

In [6]:
df["translated"].isna().value_counts()

translated
False    49778
Name: count, dtype: int64

In [7]:
tokenizer = Tokenizer(num_words=20000 , oov_token="<OOV>")
tokenizer.fit_on_texts(df["translated"])

In [8]:
tokenizer.word_index

{'<OOV>': 1,
 'the': 2,
 'to': 3,
 'and': 4,
 'i': 5,
 'you': 6,
 'a': 7,
 'in': 8,
 'data': 9,
 'for': 10,
 'of': 11,
 'we': 12,
 'this': 13,
 'your': 14,
 'would': 15,
 'have': 16,
 'on': 17,
 'be': 18,
 'support': 19,
 'our': 20,
 'could': 21,
 'with': 22,
 'is': 23,
 'problem': 24,
 'that': 25,
 'are': 26,
 'customer': 27,
 'issue': 28,
 'software': 29,
 'information': 30,
 'please': 31,
 'am': 32,
 'it': 33,
 'provide': 34,
 'as': 35,
 'security': 36,
 'due': 37,
 'has': 38,
 'been': 39,
 'about': 40,
 'or': 41,
 'system': 42,
 'appreciate': 43,
 'if': 44,
 'medical': 45,
 'but': 46,
 'assistance': 47,
 'integration': 48,
 'any': 49,
 'investment': 50,
 'dear': 51,
 'tools': 52,
 'me': 53,
 'an': 54,
 'can': 55,
 'n': 56,
 'digital': 57,
 'help': 58,
 'project': 59,
 'thank': 60,
 'which': 61,
 'resolve': 62,
 'forward': 63,
 'look': 64,
 'team': 65,
 'updates': 66,
 'access': 67,
 'my': 68,
 'marketing': 69,
 'us': 70,
 'more': 71,
 'issues': 72,
 'management': 73,
 'might': 74,


In [9]:
sequences = tokenizer.texts_to_sequences(df["translated"])
sequences

[[51,
  19,
  65,
  56,
  299,
  15,
  76,
  3,
  121,
  7,
  694,
  36,
  285,
  25,
  185,
  441,
  194,
  1141,
  11,
  20,
  412,
  140,
  139,
  305,
  3803,
  3897,
  4,
  587,
  212,
  17,
  477,
  211,
  2,
  891,
  10,
  2,
  4258,
  23,
  25,
  2,
  285,
  23,
  7,
  129,
  9,
  163,
  165,
  3,
  7,
  838,
  1070,
  61,
  2543,
  7,
  256,
  504,
  10,
  137,
  30,
  4,
  2,
  730,
  348,
  806,
  11,
  20,
  475,
  56,
  1310,
  291,
  901,
  16,
  2252,
  1521,
  748,
  4,
  2113,
  8,
  2,
  139,
  105,
  2,
  377,
  11,
  20,
  4444,
  2544,
  4,
  4445,
  110,
  2,
  1196,
  38,
  98,
  39,
  1963,
  5415,
  271,
  357],
 [51,
  27,
  19,
  65,
  56,
  326,
  32,
  151,
  3,
  121,
  7,
  256,
  24,
  22,
  2,
  3545,
  334,
  73,
  1974,
  61,
  185,
  589,
  3,
  18,
  2803,
  13,
  581,
  23,
  1687,
  67,
  3,
  334,
  132,
  413,
  3,
  1188,
  905,
  5,
  16,
  192,
  3,
  907,
  8,
  204,
  361,
  183,
  375,
  1515,
  4,
  139,
  46,
  2,
  28,
  99,
  56,
  932

In [10]:
padded_sequence = pad_sequences(sequences , maxlen=500 ,padding="post")
padded_sequence

array([[  51,   19,   65, ...,    0,    0,    0],
       [  51,   27,   19, ...,    0,    0,    0],
       [  51,   27,   19, ...,    0,    0,    0],
       ...,
       [ 176,   27,   19, ...,    0,    0,    0],
       [   2, 1157,   11, ...,    0,    0,    0],
       [  51,   27,   19, ...,    0,    0,    0]],
      shape=(49778, 500), dtype=int32)

In [30]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Attention, Dense, Dropout,GlobalAveragePooling1D
import tensorflow as tf
input_seq = Input(shape=(500,))

# Embedding
x = Embedding(input_dim=20000, output_dim=128, input_length=500, mask_zero=True)(input_seq)

# BiLSTM
x = Bidirectional(LSTM(128, return_sequences=True))(x)

# Attention
att = Attention()([x, x])  
x = GlobalAveragePooling1D()(att)
x = Dropout(0.5)(x)


# Output layer
output = Dense(3, activation="softmax")(x)  

from tensorflow.keras.models import Model
model = Model(inputs=input_seq, outputs=output)

model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 500, 128)  │  2,560,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 500)       │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 500, 256)  │    263,168 │ embedding_3[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_3         │ (None, 500, 256)  │          0 │ bidirectional_3[… │
│ (Attention)         │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convert_to_tensor_3 │ (None, 500)       │          0 │ not_equal_3[0][0] │
│ (ConvertToTensor)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ attention_3[0][0… │
│ (GlobalAveragePool… │                   │            │ convert_to_tenso… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        771 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,823,939 (10.77 MB)

 Trainable params: 2,823,939 (10.77 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [32]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(padded_sequence, 
                                                  df["lables"].values,
                                                  test_size=0.1,
                                                  random_state=42)


In [33]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    batch_size=64,
                    epochs=10,   
                    callbacks=[early_stop])


Epoch 1/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1405s 2s/step - accuracy: 0.6513 - loss: 0.7049 - val_accuracy: 0.6848 - val_loss: 0.6477
Epoch 2/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1199s 2s/step - accuracy: 0.7166 - loss: 0.5878 - val_accuracy: 0.6997 - val_loss: 0.6149
Epoch 3/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1102s 2s/step - accuracy: 0.7462 - loss: 0.5329 - val_accuracy: 0.7224 - val_loss: 0.6020
Epoch 4/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1033s 1s/step - accuracy: 0.7743 - loss: 0.4836 - val_accuracy: 0.7274 - val_loss: 0.6065
Epoch 5/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1019s 1s/step - accuracy: 0.8012 - loss: 0.4342 - val_accuracy: 0.7463 - val_loss: 0.5999
Epoch 6/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1021s 1s/step - accuracy: 0.8282 - loss: 0.3834 - val_accuracy: 0.7531 - val_loss: 0.6162
Epoch 7/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1024s 1s/step - accuracy: 0.8510 - loss: 0.3366 - val_accuracy: 0.7628 - val_loss: 0.6470
Epoch 8/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1387s 2s/step - accuracy: 0.8732 - loss: 0.2931 - 

In [34]:
model.save("bilstm_attention_ticket_classifier.keras")

In [35]:
tick = "hello world i am good"
tick1 = tick.split()
print(tick1)

['hello', 'world', 'i', 'am', 'good']


In [37]:
print(len(tokenizer.word_index))

9550
